# Çatı Güneş Paneli — Segmentasyon Deney Defteri
### Evka-5 Mahallesi, Çiğli / İzmir · pilot bölge

Bu defter, deney protokolündeki şu deneyleri koşar:

| Deney | Soru | Bu defterdeki bölüm |
|---|---|---|
| **D1** | SAM'e çatıyı nasıl göstereceğiz? | 5 — üç yönlendirme stratejisi |
| **D2** | Hangi SAM varyantı, doğruluk mu hız mı? | 6 — ViT-H / ViT-B / MobileSAM |
| **D5** | Azimut görüntüden çıkarılabilir mi? | 8 — mahya + **180° belirsizlik** |
| **D7** | PVGIS İzmir'de ne veriyor? | 9 — üretim hesabı |

**Yerel makinede zaten doğrulanan sonuçlar** (bu defter bunları GPU'da tekrarlayıp genişletir):
- MobileSAM ONNX, CPU'da bina başına 244 ms, 202 bina 49 saniye
- SAM maskesi / MS ayak izi oranı: medyan 1,21 · MAPE medyan %30,9
- **Büyük binalarda 0,99 (çok iyi), küçük binalarda 1,56 (kötü)** — asıl bulgu
- Maske–ayak izi merkez kayması medyan 1,38 m (hizalama sorunu yok)

> ⚠️ **Görüntü kaynağı uyarısı (D0).** Aşağıda Esri World Imagery *teknik deneme* amacıyla
> kullanılıyor. Teze girmeden önce lisans şartları okunmalı; toplu indirme ve türev model
> eğitimi her sağlayıcıda serbest değildir.

## 0. Kurulum ve donanım

In [ ]:
!pip -q install segment-anything onnxruntime-gpu shapely pyproj rasterio 2>/dev/null || \
 pip -q install segment-anything onnxruntime shapely pyproj

import torch, sys, platform
print("python  :", sys.version.split()[0])
print("torch   :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print(">>> GPU YOK. Colab'da: Runtime > Change runtime type > T4 GPU")
DEV = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os, json, math, time, io, gzip, csv, urllib.parse
import numpy as np, requests
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import matplotlib.pyplot as plt

os.makedirs("veri", exist_ok=True); os.makedirs("modeller", exist_ok=True)
UA = {"User-Agent": "YBS-bitirme-cati-gunes/1.0 (akademik arastirma)"}
print("hazir")

## 1. Çalışma alanı

Mahalle sınırını OSM'den çekiyoruz. **Başka bir mahalle denemek için tek satır değiştirin.**
Not: "Evka 5" Bornova'da değil **Çiğli'de**; Bornova'dakiler Evka 3 ve Evka 4.

In [ ]:
MAHALLE = "EVKA 5 Mahallesi Izmir"     # <-- degistirilecek tek yer

u = "https://nominatim.openstreetmap.org/search?" + urllib.parse.urlencode(
    {"q": MAHALLE, "format": "json", "polygon_geojson": 1, "limit": 5, "countrycodes": "tr"})
sonuc = requests.get(u, headers=UA, timeout=60).json()
alan = next(s for s in sonuc if s.get("class") == "boundary")
print(alan["display_name"])
print("OSM:", alan["osm_type"], alan["osm_id"])

from shapely.geometry import shape, Point, Polygon
from shapely.ops import transform
from pyproj import Transformer

SINIR = shape(alan["geojson"])
LON0, LAT0, LON1, LAT1 = SINIR.bounds
to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32635", always_xy=True).transform  # UTM 35N
print(f"bbox   lat {LAT0:.6f}..{LAT1:.6f}   lon {LON0:.6f}..{LON1:.6f}")
print(f"alan   {transform(to_utm, SINIR).area/1e6:.3f} km2")
json.dump(alan["geojson"], open("veri/sinir.geojson", "w"))

## 2. Bina ayak izleri — iki kaynak

**Microsoft Global ML Building Footprints**: Türkiye 386 karo ile kapsanıyor, İzmir'in tamamı
tek karoda (`122100310`). Makine üretimi, homojen — ama İzmir'de **eksik bina** sorunu var.

**OSM**: daha eksiksiz görünüyor ama kapsaması mahalleden mahalleye 6 kata kadar değişiyor.
Yerelde bu karşılaştırma Overpass hız sınırına takıldı; Colab'ın IP'sinden genelde geçer.

In [ ]:
def quadkey(lat, lon, z=9):
    s = math.sin(math.radians(lat)); n = 1 << z
    x = int((lon + 180) / 360 * n)
    y = int((0.5 - math.log((1+s)/(1-s)) / (4*math.pi)) * n)
    return "".join(str((1 if x & (1<<(i-1)) else 0) + (2 if y & (1<<(i-1)) else 0))
                   for i in range(z, 0, -1))

idx = "https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv"
rows = list(csv.DictReader(io.StringIO(requests.get(idx, headers=UA, timeout=180).text)))
qk = quadkey((LAT0+LAT1)/2, (LON0+LON1)/2)
row = next(r for r in rows if r["QuadKey"] == qk and r["Location"] == "Turkey")
print("karo", qk, "|", row["Size"], "|", row["UploadDate"])

raw = requests.get(row["Url"], headers=UA, timeout=600).content
try:    txt = gzip.decompress(raw).decode()
except: txt = raw.decode()
print("karodaki toplam bina:", txt.count("\n"))

In [ ]:
ms = []
for line in txt.strip().split("\n"):
    f = json.loads(line); c = f["geometry"]["coordinates"][0]
    lon = sum(p[0] for p in c)/len(c); lat = sum(p[1] for p in c)/len(c)
    if LON0 <= lon <= LON1 and LAT0 <= lat <= LAT1 and SINIR.contains(Point(lon, lat)):
        ms.append(f)
ms_alan = np.array([transform(to_utm, shape(f["geometry"])).area for f in ms])
print(f"MS binasi        : {len(ms)}")
print(f"  medyan ayak izi: {np.median(ms_alan):.1f} m2")
print(f"  toplam         : {ms_alan.sum()/1e4:.2f} ha "
      f"(mahallenin %{100*ms_alan.sum()/transform(to_utm,SINIR).area:.1f}'i)")
json.dump({"type":"FeatureCollection","features":ms}, open("veri/ms_binalar.geojson","w"))
del txt, raw

In [ ]:
# --- OSM ayak izleri (POST degil GET; Overpass POST'u bazi aglarda bloklu) ---
def overpass(q, tries=4, bekle=20):
    for i in range(tries):
        try:
            r = requests.get("https://overpass-api.de/api/interpreter?data=" + urllib.parse.quote(q),
                             headers=UA, timeout=180)
            if r.status_code == 200: return r.json()
            print("  HTTP", r.status_code, "- tekrar")
        except Exception as e:
            print(" ", type(e).__name__, "- tekrar")
        time.sleep(bekle)
    return None

d = overpass(f"[out:json][timeout:150];way[building]({LAT0},{LON0},{LAT1},{LON1});out geom;")
osm = []
if d:
    for e in d["elements"]:
        g = e.get("geometry")
        if not g or len(g) < 4: continue
        ring = [(p["lon"], p["lat"]) for p in g]
        try: poly = Polygon(ring)
        except Exception: continue
        if poly.is_valid and not poly.is_empty and SINIR.contains(poly.centroid):
            osm.append({"type":"Feature","properties":e.get("tags",{}),
                        "geometry":{"type":"Polygon","coordinates":[ring]}})
    osm_alan = np.array([transform(to_utm, shape(f["geometry"])).area for f in osm])
    print(f"\nOSM binasi : {len(osm)}   medyan {np.median(osm_alan):.1f} m2   "
          f"toplam {osm_alan.sum()/1e4:.2f} ha")
    print(f"OSM / MS   : {len(osm)/len(ms):.2f}x bina   {osm_alan.sum()/ms_alan.sum():.2f}x alan")
    json.dump({"type":"FeatureCollection","features":osm}, open("veri/osm_binalar.geojson","w"))
else:
    print("Overpass alinamadi - MS ile devam edilecek")

## 3. Görüntü — Esri World Imagery karoları

z19 ≈ **0,234 m/piksel**. Kıyas: INRIA veri seti 0,3 m/px, RID Google Satellite.
Evka-5 için 374 karo → 4352×5632 px mozaik.

In [ ]:
Z, TS = 19, 256
TILE = "https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"

def deg2tile(lat, lon, z):
    n = 1 << z
    return (int((lon+180)/360*n),
            int((1 - math.asinh(math.tan(math.radians(lat)))/math.pi)/2*n))

def tile2deg(x, y, z):
    n = 1 << z
    return (math.degrees(math.atan(math.sinh(math.pi*(1-2*y/n)))), x/n*360-180)

x0, y0 = deg2tile(LAT1, LON0, Z)      # kuzeybati
x1, y1 = deg2tile(LAT0, LON1, Z)      # guneydogu
XS, YS = range(x0, x1+1), range(y0, y1+1)
print(f"{len(XS)} x {len(YS)} = {len(XS)*len(YS)} karo")

import concurrent.futures as cf
def cek(xy):
    x, y = xy; p = f"veri/t_{Z}_{x}_{y}.jpg"
    if os.path.exists(p) and os.path.getsize(p) > 1000: return True
    try:
        r = requests.get(TILE.format(z=Z, x=x, y=y), headers=UA, timeout=45)
        if r.status_code == 200 and len(r.content) > 1000:
            open(p, "wb").write(r.content); return True
    except Exception: pass
    return False

isler = [(x, y) for y in YS for x in XS]
with cf.ThreadPoolExecutor(max_workers=12) as ex:
    ok = sum(ex.map(cek, isler))
print(f"indirilen: {ok}/{len(isler)}")

In [ ]:
W, H = len(XS)*TS, len(YS)*TS
MOZ = Image.new("RGB", (W, H))
for j, y in enumerate(YS):
    for i, x in enumerate(XS):
        p = f"veri/t_{Z}_{x}_{y}.jpg"
        if os.path.exists(p):
            try: MOZ.paste(Image.open(p), (i*TS, j*TS))
            except Exception: pass
MOZ.save("veri/mozaik.jpg", quality=92)

MPP = 156543.03392 * math.cos(math.radians((LAT0+LAT1)/2)) / (1 << Z)
print(f"mozaik {W}x{H} px | {MPP:.3f} m/piksel | {W*MPP:.0f}x{H*MPP:.0f} m")

def lonlat2px(lon, lat):
    n = (1 << Z) * TS
    return ((lon+180)/360*n - x0*TS,
            (1 - math.asinh(math.tan(math.radians(lat)))/math.pi)/2*n - y0*TS)

def px2lonlat(px, py):
    n = (1 << Z) * TS
    gx, gy = px + x0*TS, py + y0*TS
    return (gx/n*360-180, math.degrees(math.atan(math.sinh(math.pi*(1-2*gy/n)))))

plt.figure(figsize=(9, 12)); plt.imshow(MOZ.resize((W//6, H//6))); plt.axis("off")
plt.title(f"{alan['display_name'].split(',')[0]} — {MPP:.2f} m/px"); plt.show()

## 4. Modeller

İki model indiriyoruz:
- **SAM ViT-B** (375 MB, PyTorch) — GPU'da tam sürüm, `segment-anything` kütüphanesi
- **MobileSAM ONNX** (28+16 MB) — CPU'da bile koşan hafif sürüm

ViT-H de eklenebilir (2,4 GB) ama Colab'ın ücretsiz diskinde ve T4'ün belleğinde
ViT-B ile başlamak daha akıllı. D2 deneyi tam olarak bu üçünü karşılaştırmak.

In [ ]:
MODELLER = {
 "sam_vit_b.pth"      : "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
 "msam_encoder.onnx"  : "https://huggingface.co/Acly/MobileSAM/resolve/main/mobile_sam_image_encoder.onnx",
 "msam_decoder.onnx"  : "https://huggingface.co/Acly/MobileSAM/resolve/main/sam_mask_decoder_single.onnx",
}
for ad, url in MODELLER.items():
    p = "modeller/" + ad
    if os.path.exists(p) and os.path.getsize(p) > 1e6:
        print(f"{ad:22s} var  {os.path.getsize(p)/1e6:6.1f} MB"); continue
    r = requests.get(url, headers=UA, timeout=900, stream=True); n = 0
    with open(p, "wb") as o:
        for ch in r.iter_content(1 << 20): o.write(ch); n += len(ch)
    print(f"{ad:22s} indi {n/1e6:6.1f} MB")

In [ ]:
# --- SAM ViT-B (PyTorch, GPU) ---
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator
sam = sam_model_registry["vit_b"]("modeller/sam_vit_b.pth").to(DEV).eval()
PRED = SamPredictor(sam)
print("SAM ViT-B yuklendi ->", DEV)

# --- MobileSAM (ONNX) ---
import onnxruntime as ort
MEAN = np.array([123.675, 116.28, 103.53], np.float32)
STD  = np.array([58.395, 57.12, 57.375],  np.float32)

class MobileSAM:
    def __init__(self):
        pr = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
             if "CUDAExecutionProvider" in ort.get_available_providers() else ["CPUExecutionProvider"]
        self.enc = ort.InferenceSession("modeller/msam_encoder.onnx", providers=pr)
        self.dec = ort.InferenceSession("modeller/msam_decoder.onnx", providers=pr)
        self.ad  = self.enc.get_inputs()[0].name
    def embed(self, rgb):
        return self.enc.run(None, {self.ad: (rgb.astype(np.float32)-MEAN)/STD})[0]
    def box(self, emb, b, hw):
        # SAM kutu-yonlendirmesi: iki nokta, etiket 2 (sol-ust) ve 3 (sag-alt)
        out = self.dec.run(None, {
            "image_embeddings": emb,
            "point_coords": np.array([[[b[0], b[1]], [b[2], b[3]]]], np.float32),
            "point_labels": np.array([[2.0, 3.0]], np.float32),
            "mask_input":  np.zeros((1,1,256,256), np.float32),
            "has_mask_input": np.zeros(1, np.float32),
            "orig_im_size": np.array(hw, np.float32)})
        return out[0][0,0] > 0.0, float(np.ravel(out[1])[0])

MSAM = MobileSAM()
print("MobileSAM ONNX yuklendi ->", ort.get_available_providers()[0])

## 5. D1 — SAM'e çatıyı nasıl göstereceğiz?

Protokoldeki üç strateji, aynı alanda, aynı metrikle:

| | Strateji | Not |
|---|---|---|
| **A** | Otomatik maske üretici + geometrik filtre | SAM sınıf bilmez: bir karoda 300+ maske çıkarır, hangisi çatı belli değil |
| **B** | **Ayak izi ile kutu-yönlendirmesi** | "Hangisi çatı" sorusunu ortadan kaldırır — *önerilen* |
| **C** | Sadece ayak izi, SAM yok | **Temel çizgi.** SAM gerçekten katkı sağlıyor mu? |

In [ ]:
# Binalarin mozaikteki piksel kutulari
KAYNAK = ms                      # "osm" da denenebilir
kutular, ayak_alanlari = [], []
for f in KAYNAK:
    px = [lonlat2px(p[0], p[1]) for p in f["geometry"]["coordinates"][0]]
    X = [p[0] for p in px]; Y = [p[1] for p in px]
    kutular.append((min(X), min(Y), max(X), max(Y)))
    ayak_alanlari.append(transform(to_utm, shape(f["geometry"])).area)
kutular = np.array(kutular); ayak_alanlari = np.array(ayak_alanlari)
print(f"{len(kutular)} bina | kutu medyan {np.median(kutular[:,2]-kutular[:,0]):.0f}x"
      f"{np.median(kutular[:,3]-kutular[:,1]):.0f} px")

T, ORT = 1024, 256                # kare boyu, ortusme
step = T - ORT
kareler = [(min(x, W-T), min(y, H-T))
           for y in range(0, max(H-T,0)+step, step)
           for x in range(0, max(W-T,0)+step, step)]

def kareye_ata(kutular):
    # Her binayi, tamamen icinde kaldigi ve merkezine en yakin kareye atar
    ata = {}
    for i, b in enumerate(kutular):
        bx, by = (b[0]+b[2])/2, (b[1]+b[3])/2
        for (tx, ty) in kareler:
            if b[0] > tx+8 and b[1] > ty+8 and b[2] < tx+T-8 and b[3] < ty+T-8:
                d = (bx-(tx+T/2))**2 + (by-(ty+T/2))**2
                if i not in ata or d < ata[i][1]: ata[i] = ((tx, ty), d)
    g = {}
    for i, (t, _) in ata.items(): g.setdefault(t, []).append(i)
    return g

GRUP = kareye_ata(kutular)
print(f"{sum(len(v) for v in GRUP.values())}/{len(kutular)} bina "
      f"{len(GRUP)} kareye atandi")

### D1-B · Ayak izi ile kutu-yönlendirmesi (SAM ViT-B, GPU)

In [ ]:
import cv2

def geometri(mask, mpp):
    # Maskenin minimum donmus dikdortgeninden mahya yonu ve kenarlar
    c, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not c: return None
    big = max(c, key=cv2.contourArea)
    (cx, cy), (w, h), ang = cv2.minAreaRect(big)
    if h > w: ang += 90.0
    uzun, kisa = max(w, h), min(w, h)
    return dict(mahya=ang % 180.0, uzun=uzun*mpp, kisa=kisa*mpp,
                doluluk=float(cv2.contourArea(big)/max(uzun*kisa, 1)))

def kosu_vitb():
    sonuc = []; t0 = time.time(); enc_s = 0.0
    for n, ((tx, ty), idx) in enumerate(sorted(GRUP.items()), 1):
        crop = np.array(MOZ.crop((tx, ty, tx+T, ty+T)))
        te = time.time(); PRED.set_image(crop); enc_s += time.time()-te
        # butun kutulari tek seferde ver -> GPU'da cok daha hizli
        bx = torch.tensor([[kutular[i][0]-tx, kutular[i][1]-ty,
                            kutular[i][2]-tx, kutular[i][3]-ty] for i in idx],
                          dtype=torch.float, device=DEV)
        tb = PRED.transform.apply_boxes_torch(bx, (T, T))
        masks, iou, _ = PRED.predict_torch(point_coords=None, point_labels=None,
                                           boxes=tb, multimask_output=False)
        masks = masks[:, 0].cpu().numpy(); iou = iou[:, 0].cpu().numpy()
        for k, i in enumerate(idx):
            m = masks[k]
            if m.sum() < 40: continue
            ys_, xs_ = np.nonzero(m)
            lon, lat = px2lonlat(xs_.mean()+tx, ys_.mean()+ty)
            r = dict(i=int(i), sam_alan=float(m.sum())*MPP**2,
                     ayak_alan=float(ayak_alanlari[i]), iou=float(iou[k]),
                     lon=lon, lat=lat,
                     dx=float(xs_.mean()+tx-(kutular[i][0]+kutular[i][2])/2),
                     dy=float(ys_.mean()+ty-(kutular[i][1]+kutular[i][3])/2))
            g = geometri(m, MPP)
            if g: r.update(g)
            sonuc.append(r)
        if n % 5 == 0 or n == len(GRUP):
            print(f"  {n}/{len(GRUP)} kare · {len(sonuc)} bina · {time.time()-t0:.0f}s")
    print(f"\nViT-B TOPLAM {time.time()-t0:.1f} s | kodlayici {enc_s:.1f} s | "
          f"bina basina {(time.time()-t0)/max(len(sonuc),1)*1000:.0f} ms")
    return sonuc, time.time()-t0

VITB, SURE_VITB = kosu_vitb()
json.dump(VITB, open("veri/sonuc_vitb.json", "w"))

## 6. D2 — Model karşılaştırması: doğruluk mu, hız mı?

In [ ]:
def kosu_msam():
    sonuc = []; t0 = time.time()
    for (tx, ty), idx in sorted(GRUP.items()):
        crop = np.array(MOZ.crop((tx, ty, tx+T, ty+T)))
        emb = MSAM.embed(crop)
        for i in idx:
            b = kutular[i]
            m, iou = MSAM.box(emb, (b[0]-tx, b[1]-ty, b[2]-tx, b[3]-ty), (T, T))
            if m.sum() < 40: continue
            sonuc.append(dict(i=int(i), sam_alan=float(m.sum())*MPP**2,
                              ayak_alan=float(ayak_alanlari[i]), iou=iou))
    return sonuc, time.time()-t0

MSAM_R, SURE_MSAM = kosu_msam()
print(f"MobileSAM: {len(MSAM_R)} bina, {SURE_MSAM:.1f} s, "
      f"{SURE_MSAM/max(len(MSAM_R),1)*1000:.0f} ms/bina")

import pandas as pd
def ozet(ad, r, sure):
    sa = np.array([x["sam_alan"] for x in r]); fa = np.array([x["ayak_alan"] for x in r])
    mape = np.abs(sa-fa)/fa
    return dict(model=ad, bina=len(r), sure_s=round(sure, 1),
                ms_bina=round(sure/max(len(r), 1)*1000),
                oran_medyan=round(float(np.median(sa/fa)), 3),
                MAPE_medyan=round(float(np.median(mape))*100, 1),
                hata_10=int((mape < 0.10).sum()), hata_20=int((mape < 0.20).sum()),
                iou_medyan=round(float(np.median([x["iou"] for x in r])), 3))

TABLO = pd.DataFrame([ozet("SAM ViT-B", VITB, SURE_VITB),
                      ozet("MobileSAM", MSAM_R, SURE_MSAM)])
display(TABLO)
TABLO.to_csv("veri/d2_model_karsilastirma.csv", index=False)
print("\nYerel CPU referansi: MobileSAM 244 ms/bina (202 bina, 49 s)")

### D1-A · Otomatik maske üretici — SAM'in çatı bilmediğini göstermek

Bu hücre tek bir kareyi otomatik maske üreticisine verir. Çıkan maske sayısı,
"SAM'e ne soracağımızı seçmek, modeli seçmekten önemli" iddiasının kanıtıdır.

In [ ]:
(tx, ty), idx = sorted(GRUP.items(), key=lambda kv: -len(kv[1]))[0]
crop = np.array(MOZ.crop((tx, ty, tx+T, ty+T)))

AMG = SamAutomaticMaskGenerator(sam, points_per_side=32, pred_iou_thresh=0.88,
                                stability_score_thresh=0.92, min_mask_region_area=200)
t0 = time.time(); hepsi = AMG.generate(crop); t_amg = time.time()-t0
print(f"otomatik maske: {len(hepsi)} adet, {t_amg:.1f} s")
print(f"ayni karede gercek bina sayisi: {len(idx)}")
print(f"-> her bina icin ~{len(hepsi)/max(len(idx),1):.0f} maske elenmeli\n")

# geometrik filtre denemesi: alan + dikdortgensellik + en-boy
kalan = []
for m in hepsi:
    a = m["area"] * MPP**2
    if not (60 < a < 3000): continue
    g = geometri(m["segmentation"], MPP)
    if g and g["doluluk"] > 0.65 and g["uzun"]/max(g["kisa"], 1e-6) < 4:
        kalan.append(m)
print(f"alan + dikdortgensellik filtresinden sonra: {len(kalan)} maske (gercek {len(idx)})")

fig, ax = plt.subplots(1, 3, figsize=(19, 6.6))
ax[0].imshow(crop); ax[0].set_title(f"ham kare ({len(idx)} bina)")
rng = np.random.default_rng(0)
for a_, veri, ad in ((ax[1], hepsi, f"D1-A ham: {len(hepsi)} maske"),
                     (ax[2], kalan, f"D1-A filtreli: {len(kalan)} maske")):
    ov = crop.astype(np.float32).copy()
    for m in veri:
        ov[m["segmentation"]] = ov[m["segmentation"]]*0.45 + rng.integers(60, 255, 3)*0.55
    a_.imshow(np.clip(ov, 0, 255).astype(np.uint8)); a_.set_title(ad)
for a_ in ax: a_.axis("off")
plt.tight_layout(); plt.show()

## 7. Metrikler — alan, hizalama, boyuta göre kırılım

In [ ]:
sa = np.array([x["sam_alan"] for x in VITB]); fa = np.array([x["ayak_alan"] for x in VITB])
oran = sa/fa; mape = np.abs(sa-fa)/fa
print("=== D1-B · SAM ViT-B vs MS ayak izi ===")
print(f"  oran       medyan {np.median(oran):.3f}  %25 {np.percentile(oran,25):.2f}  %75 {np.percentile(oran,75):.2f}")
print(f"  ALAN MAPE  medyan %{100*np.median(mape):.1f}   ortalama %{100*mape.mean():.1f}")
for e in (0.10, 0.20, 0.50):
    print(f"    |hata| < %{int(e*100):2d}: {(mape<e).sum():3d}/{len(sa)}  (%{100*(mape<e).mean():.0f})")

buyuk = fa > np.median(fa)
print(f"\n  BOYUTA GORE  buyuk binalar {np.median(oran[buyuk]):.2f}x | "
      f"kucuk binalar {np.median(oran[~buyuk]):.2f}x")
print("
  KIYAS  yerel MobileSAM (CPU): oran 1.21 | buyuk 0.99 | kucuk 1.56")
print("         Colab ViT-B (GPU)    : oran 0.68 | buyuk 0.58 | kucuk 0.79")
print("  -> Iki varyant TERS yonde sapiyor. Hangisinin dogru oldugunu")
print("     ancak elle etiketlenmis altin veri seti soyler.")

d = np.array([[x["dx"], x["dy"]] for x in VITB]); kay = np.hypot(d[:, 0], d[:, 1])*MPP
print(f"\n  MERKEZ KAYMASI  medyan {np.median(kay):.2f} m | ort {kay.mean():.2f} m | "
      f">10 m: {(kay>10).sum()}/{len(kay)}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
ax[0].hist(oran, bins=40, range=(0, 3), color="#a8650a"); ax[0].axvline(1, color="k", ls="--")
ax[0].set_title("SAM alani / ayak izi alani"); ax[0].set_xlabel("oran")
ax[1].scatter(fa, sa, s=13, alpha=.6, color="#1a6a73")
lim = [0, float(np.percentile(fa, 99))*1.3]; ax[1].plot(lim, lim, "k--", lw=1)
ax[1].set_xlim(lim); ax[1].set_ylim(lim)
ax[1].set_xlabel("ayak izi alani (m2)"); ax[1].set_ylabel("SAM alani (m2)")
ax[1].set_title("kesikli cizgi = tam uyum")
ax[2].hist(kay, bins=40, color="#9c3521"); ax[2].set_xlabel("merkez kaymasi (m)")
ax[2].set_title("maske ile ayak izi kaymasi")
plt.tight_layout(); plt.show()

## 8. D5 — Azimut

> ⚠️ **Buraya dikkat — burada kolayca yanlış sonuç üretiliyor.**
>
> İlk denememde "binaların %68'i güneye bakıyor" çıktı. Bu bir bulgu değildi:
> kod, iki zıt yönden güneye yakın olanı seçtiği için sonuç kendiliğinden güneye
> yaslıyordu.
>
> **Dik (nadir) görüntüde beşik çatının iki yüzü de benzer görünür.** Mahya çizgisi
> tek bir *eksen* verir; çatı yüzeyleri bu eksenin 90° sağında **ve** solundadır. Yani
> her bina için **180° belirsizlik** kalır ve bu görüntüden çözülemez — gölge analizi
> veya yükseklik verisi gerekir. D6 ile aynı engel.
>
> Bu yüzden aşağıda sadece **mahya yönü** raporlanıyor, "güneye bakıyor" iddiası değil.

In [ ]:
mahya = np.array([x["mahya"] for x in VITB if "mahya" in x])
a = np.deg2rad(2*mahya)                      # eksenel veri: aciyi 2 ile carp
R = float(np.hypot(np.cos(a).mean(), np.sin(a).mean()))
baskin = (np.rad2deg(np.arctan2(np.sin(a).mean(), np.cos(a).mean()))/2) % 180
print(f"mahya yonu · {len(mahya)} bina   (0 = dogu-bati, 90 = kuzey-guney)")
print(f"  yon tutarliligi R = {R:.3f}   (0 = rastgele, 1 = hepsi ayni yonde)")
print(f"  baskin mahya yonu = {baskin:.0f} derece")
print("  ->", "Mahalle planli, belirgin dogrultu var" if R > 0.3 else "Belirgin dogrultu yok")
print("  (yerel referans: R = 0.414, baskin yon 76 derece)")

dol = np.array([x["doluluk"] for x in VITB if "doluluk" in x])
uz = np.array([x["uzun"] for x in VITB if "uzun" in x])
ks = np.array([x["kisa"] for x in VITB if "kisa" in x])
print(f"\n  dikdortgensellik medyan {np.median(dol):.3f}   (1.0 = tam dikdortgen)")
print(f"  uzun kenar {np.median(uz):.1f} m | kisa kenar {np.median(ks):.1f} m | "
      f"en-boy {np.median(uz/ks):.2f}")

fig = plt.figure(figsize=(11, 4.6))
axp = fig.add_subplot(121, projection="polar")
th = np.deg2rad(np.concatenate([mahya, mahya+180]))   # eksen: iki yonu de ciz
axp.hist(th, bins=36, color="#1a6a73")
axp.set_theta_zero_location("N"); axp.set_theta_direction(-1)
axp.set_title(f"mahya yonu gulu (R={R:.2f})")
ax2 = fig.add_subplot(122); ax2.hist(dol, bins=30, color="#a8650a")
ax2.set_xlabel("doluluk (maske / min. donmus dikdortgen)")
ax2.set_title("cati ne kadar dikdortgen?")
plt.tight_layout(); plt.show()

## 9. D7 — PVGIS ile üretim potansiyeli

PVGIS hesabı **sunucu tarafında** yapar ve doğrudan yıllık kWh döndürür — pvlib kurmaya
gerek yok, API anahtarı da istemiyor.

**Eğim (tilt) hâlâ elimizde yok** (D6). Bu yüzden nokta tahmin yerine **senaryo bandı**
üretiyoruz: düz çatı (0°/15°) ve eğimli çatı (30°) varsayımlarıyla üretim aralığı.
Belirsizliği gizlemek yerine ölçmek yöntemi zayıflatmaz, güçlendirir.

In [ ]:
PVGIS = "https://re.jrc.ec.europa.eu/api/v5_3/PVcalc"

def pvgis(lat, lon, kwp=1.0, kayip=14, egim=30, azimut=0):
    # azimut: 0 = guney, -90 = dogu, +90 = bati  (PVGIS kendi sozlesmesi)
    r = requests.get(PVGIS, params=dict(lat=round(lat, 4), lon=round(lon, 4),
                                        peakpower=kwp, loss=kayip, angle=egim,
                                        aspect=azimut, outputformat="json",
                                        pvtechchoice="crystSi", mountingplace="building"),
                     headers=UA, timeout=90)
    r.raise_for_status()
    t = r.json()["outputs"]["totals"]["fixed"]
    return t["E_y"], t["H(i)_y"]        # yillik kWh, yillik dusen isinim kWh/m2

# Isinim mahalle icinde neredeyse sabit -> her bina icin ayri sorgu ATMIYORUZ.
# Mahalle merkezi icin senaryo basina tek sorgu yeter: binlerce cagri yerine birkac tane.
merkez = SINIR.centroid
SENARYO = {"duz cati (0 derece)": 0, "hafif egimli (15 derece)": 15, "besik cati (30 derece)": 30}
BIRIM = {}
for ad, eg in SENARYO.items():
    e_y, h_y = pvgis(merkez.y, merkez.x, kwp=1.0, egim=eg, azimut=0)
    BIRIM[ad] = e_y
    print(f"{ad:26s} {e_y:7.1f} kWh/kWp/yil   (dusen isinim {h_y:.0f} kWh/m2)")
print(f"\nsenaryolar arasi fark: %{100*(max(BIRIM.values())/min(BIRIM.values())-1):.1f}")

In [ ]:
# Cati alanindan kapasiteye: kullanilabilir oran ve m2/kWp varsayimlari
M2_PER_KWP = 6.0          # ~200 W/m2 modul
KULLANILABILIR = {"duz cati (0 derece)": 0.55,       # sira araligi + kenar bosluklari
                  "hafif egimli (15 derece)": 0.55,
                  "besik cati (30 derece)": 0.35}    # sadece bir yuz

import pandas as pd
sat = []
for x in VITB:
    alan = x["sam_alan"]
    r = dict(i=x["i"], lon=round(x["lon"], 6), lat=round(x["lat"], 6),
             sam_alan_m2=round(alan, 1), ayak_alan_m2=round(x["ayak_alan"], 1),
             mahya_derece=round(x.get("mahya", float("nan")), 1))
    for ad, oran in KULLANILABILIR.items():
        kwp = alan*oran/M2_PER_KWP
        r[f"kWp [{ad}]"] = round(kwp, 2)
        r[f"kWh_yil [{ad}]"] = round(kwp*BIRIM[ad])
    sat.append(r)
DF = pd.DataFrame(sat)
display(DF.head(10))

kol = [c for c in DF.columns if c.startswith("kWh_yil")]
print("\n=== MAHALLE TOPLAMI (senaryo bandi) ===")
for c in kol:
    print(f"  {c:34s} {DF[c].sum()/1e6:6.3f} GWh/yil   "
          f"({DF[c.replace('kWh_yil','kWp')].sum():7.0f} kWp)")
alt, ust = DF[kol].sum().min(), DF[kol].sum().max()
print(f"\n  BANT: {alt/1e6:.3f} – {ust/1e6:.3f} GWh/yil  "
      f"(belirsizlik {100*(ust/alt-1):.0f}% — kaynagi bilinmeyen cati egimi)")
DF.to_csv("veri/evka5_uretim.csv", index=False)
print("\n-> veri/evka5_uretim.csv")

## 10. Toplumsal öncelik katmanı — kamu binaları

OSM'den okul / sağlık / belediye tesislerini çekip çatı listesiyle eşleştiriyoruz.
Bu, projenin ayırt edici iddiasının çalışan en küçük hâli.

> **SEGE gelir katmanı burada yok** — `sanayi.gov.tr` bot erişimini reddediyor, dosya
> tarayıcıdan elle indirilmeli. Geldiğinde ilçe kademesi bu tabloya bir sütun olarak eklenir.

In [ ]:
q = f"""[out:json][timeout:150];
(
 nwr[amenity=school]({LAT0},{LON0},{LAT1},{LON1});
 nwr[amenity=kindergarten]({LAT0},{LON0},{LAT1},{LON1});
 nwr[amenity=hospital]({LAT0},{LON0},{LAT1},{LON1});
 nwr[amenity=clinic]({LAT0},{LON0},{LAT1},{LON1});
 nwr[healthcare]({LAT0},{LON0},{LAT1},{LON1});
 nwr[amenity=townhall]({LAT0},{LON0},{LAT1},{LON1});
 nwr[amenity=library]({LAT0},{LON0},{LAT1},{LON1});
);
out tags center;"""
dk = overpass(q)
kamu = []
if dk:
    for e in dk["elements"]:
        c = e.get("center") or {"lat": e.get("lat"), "lon": e.get("lon")}
        if c.get("lat") is None: continue
        t = e.get("tags", {})
        kamu.append(dict(ad=t.get("name", "(isimsiz)"),
                         tur=t.get("amenity") or t.get("healthcare"),
                         lat=c["lat"], lon=c["lon"]))
print(f"kamu tesisi: {len(kamu)}")

# en yakin cati ile eslestir (basit oklid; mahalle olceginde yeterli)
AGIRLIK = {"school": 1.0, "kindergarten": 1.0, "hospital": 0.9, "clinic": 0.9,
           "centre": 0.9, "townhall": 0.6, "library": 0.6}
DF["kamu_agirlik"] = 0.0
DF["kamu_ad"] = ""
for k in kamu:
    d = np.hypot((DF["lon"]-k["lon"])*np.cos(np.radians(k["lat"]))*111320,
                 (DF["lat"]-k["lat"])*111320)
    j = int(d.idxmin())
    if d[j] < 60:                                    # 60 m icinde bir cati varsa
        w = AGIRLIK.get(k["tur"], 0.5)
        if w > DF.at[j, "kamu_agirlik"]:
            DF.at[j, "kamu_agirlik"] = w; DF.at[j, "kamu_ad"] = k["ad"]
print(f"kamu binasina eslesen cati: {(DF['kamu_agirlik']>0).sum()}")

### D8 — Önceliklendirme: ağırlıkları kim belirliyor?

Üç yöntem aynı veriye uygulanır ve **ilk-20 listeleri karşılaştırılır.** Asıl teslim
edilecek çıktı sıralamanın kendisi değil, **duyarlılık analizidir**: ağırlıklar ±%20
oynatıldığında liste ne kadar değişiyor?

In [ ]:
def normalize(x):
    x = np.asarray(x, float); r = x.max()-x.min()
    return np.zeros_like(x) if r == 0 else (x-x.min())/r

ana = [c for c in DF.columns if c.startswith("kWh_yil")][1]      # orta senaryo
K = np.column_stack([normalize(DF[ana]), normalize(DF["kamu_agirlik"])])
ADLAR = ["teknik potansiyel", "kamu binasi"]

def entropi_agirlik(K):
    P = K/ (K.sum(0)+1e-12); P = np.clip(P, 1e-12, 1)
    E = -(P*np.log(P)).sum(0)/np.log(len(K))
    d = 1-E
    return d/d.sum()

YONTEM = {"AHP benzeri (0.60 / 0.40)": np.array([0.60, 0.40]),
          "esit agirlik":              np.array([0.50, 0.50]),
          "entropi":                   entropi_agirlik(K)}
SIRA = {}
for ad, w in YONTEM.items():
    s = K @ (w/w.sum())
    SIRA[ad] = np.argsort(-s)
    print(f"{ad:28s} agirliklar {np.round(w/w.sum(),3)}")

ilk20 = {ad: set(v[:20]) for ad, v in SIRA.items()}
print("\n=== ILK-20 ORTUSMESI ===")
adlar = list(ilk20)
for i in range(len(adlar)):
    for j in range(i+1, len(adlar)):
        o = len(ilk20[adlar[i]] & ilk20[adlar[j]])
        print(f"  {adlar[i]:26s} <-> {adlar[j]:26s} {o:2d}/20")

In [ ]:
def kendall_tau(a, b):
    # scipy'siz Kendall tau-b; n kucuk oldugu icin O(n^2) yeterli
    n = len(a); uy = ters = 0
    for i in range(n):
        for j in range(i+1, n):
            s = np.sign(a[i]-a[j])*np.sign(b[i]-b[j])
            if s > 0: uy += 1
            elif s < 0: ters += 1
    return (uy-ters)/max(uy+ters, 1)

w0 = YONTEM["AHP benzeri (0.60 / 0.40)"]; w0 = w0/w0.sum()
temel = K @ w0
rng = np.random.default_rng(42)
taular, ort20 = [], []
for _ in range(200):
    w = w0 * (1 + rng.uniform(-0.20, 0.20, size=w0.shape))
    s = K @ (w/w.sum())
    taular.append(kendall_tau(temel, s))
    ort20.append(len(set(np.argsort(-temel)[:20]) & set(np.argsort(-s)[:20])))
taular = np.array(taular); ort20 = np.array(ort20)
print("=== DUYARLILIK: agirliklar +-%20 oynatildiginda (200 deneme) ===")
print(f"  Kendall tau   ortalama {taular.mean():.3f}   en dusuk {taular.min():.3f}")
print(f"  ilk-20 ortusmesi  ortalama {ort20.mean():.1f}/20   en dusuk {ort20.min()}/20")
print("\n  Juri 'bu agirliklari neden sectiniz' diye sordugunda gosterilecek tablo budur.")
print("  tau 0.9 uzerindeyse siralama agirliga karsi saglam demektir.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(taular, bins=30, color="#1a6a73"); ax[0].set_xlabel("Kendall tau")
ax[0].set_title("siralama kararliligi")
ax[1].hist(ort20, bins=range(int(ort20.min()), 22), color="#a8650a")
ax[1].set_xlabel("ilk-20 ortusmesi"); ax[1].set_title("liste ne kadar degisiyor")
plt.tight_layout(); plt.show()

## 11. Sonuç haritası ve dışa aktarım

In [ ]:
en_iyi = SIRA["AHP benzeri (0.60 / 0.40)"][:20]
gorsel = np.array(MOZ, dtype=np.float32)
IYI = np.array([0, 210, 95], np.float32); KOTU = np.array([255, 70, 0], np.float32)
VUR = np.array([255, 235, 0], np.float32)
for (tx, ty), idx in sorted(GRUP.items()):
    crop = np.array(MOZ.crop((tx, ty, tx+T, ty+T))); PRED.set_image(crop)
    bx = torch.tensor([[kutular[i][0]-tx, kutular[i][1]-ty, kutular[i][2]-tx, kutular[i][3]-ty]
                       for i in idx], dtype=torch.float, device=DEV)
    tb = PRED.transform.apply_boxes_torch(bx, (T, T))
    mk, _, _ = PRED.predict_torch(point_coords=None, point_labels=None,
                                  boxes=tb, multimask_output=False)
    mk = mk[:, 0].cpu().numpy()
    for k, i in enumerate(idx):
        m = mk[k]
        if m.sum() < 40: continue
        satir = DF.index[DF["i"] == i]
        c = VUR if (len(satir) and satir[0] in en_iyi) else \
            (IYI if 0.8 <= (m.sum()*MPP**2)/ayak_alanlari[i] <= 1.25 else KOTU)
        sub = gorsel[ty:ty+T, tx:tx+T]; sub[m] = sub[m]*0.5 + c*0.5
son = Image.fromarray(np.clip(gorsel, 0, 255).astype(np.uint8))
son.save("veri/sonuc_haritasi.jpg", quality=90)
plt.figure(figsize=(11, 14)); plt.imshow(son.resize((W//5, H//5))); plt.axis("off")
plt.title("sari = ilk 20 oncelik · yesil = SAM ayak iziyle uyusuyor · turuncu = sapiyor")
plt.show()

In [ ]:
# GeoJSON disa aktarim (QGIS / Leaflet icin)
ozellikler = []
for _, r in DF.iterrows():
    ozellikler.append({"type": "Feature",
                       "geometry": {"type": "Point", "coordinates": [r["lon"], r["lat"]]},
                       "properties": {k: (None if pd.isna(v) else v) for k, v in r.items()}})
json.dump({"type": "FeatureCollection", "features": ozellikler},
          open("veri/evka5_sonuc.geojson", "w"), ensure_ascii=False)

from google.colab import files
for f in ["veri/evka5_uretim.csv", "veri/evka5_sonuc.geojson",
          "veri/d2_model_karsilastirma.csv", "veri/sonuc_haritasi.jpg"]:
    print(f, f"{os.path.getsize(f)/1e6:.2f} MB")
# files.download("veri/evka5_sonuc.geojson")   # indirmek icin yorumu kaldirin

---

## Bu defterin söyleyemedikleri

Dürüst olmak, sonuçları savunmaktan önemli. Şu anda **elimizde gerçek referans yok**:

1. **MAPE bir doğruluk değil, bir anlaşmazlık ölçüsü.** SAM ile Microsoft ayak izi
   arasındaki farkı ölçüyoruz; ikisi de gerçek değil. Hangisinin haklı olduğunu ancak
   **elle etiketlenmiş 150 bina** söyler. Bu iş yapılmadan "modelimiz %X başarılı"
   cümlesi kurulamaz.

2. **Eğim yok.** Microsoft verisinde `height` alanı var ama Türkiye'de 201.067 binanın
   sıfırı dolu. Google Solar API Türkiye'yi kapsıyorsa çözülür — ücretsiz deneme
   anahtarıyla tek bir çağrı bunu 5 dakikada söyler.

3. **Azimut yarım.** Mahya ekseni çıkıyor, çatının hangi yüze baktığı çıkmıyor (180°
   belirsizlik). Gölge analizi bir sonraki deney.

4. **SEGE yok.** Öncelik katmanı şu an sadece kamu binası bacağıyla çalışıyor; gelir
   bacağı SEGE 2022 elle indirilince eklenecek.

5. **Görüntü lisansı açık (D0).** Esri World Imagery teknik deneme için kullanıldı.
   Teze girmeden önce şartlar okunmalı, İzmir Büyükşehir'den ortofoto talep edilmeli.

## Sıradaki deneyler

- **D3** — INRIA / WHU üzerinde eğitilmiş U-Net'i aynı karolarda SAM'e karşı koştur
- **D4** — NREL DeepSolar ile mevcut panel/güneş enerjili su ısıtıcısı tespiti
  (Evka-5 görüntüsünde çatılarda koyu ızgara yapılar görünüyor)
- **D6** — Google Solar API kapsama testi
- **D9** — LLM rapor üretimi, üç model karşılaştırması